# cross-product-normal — ex2: batched unit normals + degenerate-triangle mask

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-product-normal`. Running the final beacon cell reports progress against the `Geometry: Cross-product surface normal` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Cross-product surface normal` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-product-normal`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-product-normal"
DD_SUBTOPIC = "Geometry: Cross-product surface normal"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-product surface normal — deepening refresher

For a single triangle, `n = cross(P2-P1, P3-P1)` then normalize. For a BATCH of `(N, 3, 3)` triangles (N triangles, each with 3 vertices, each vertex 3-D), the same recipe applies along the last axis:

```python
e1 = tris[:, 1] - tris[:, 0]          # (N, 3)
e2 = tris[:, 2] - tris[:, 0]          # (N, 3)
n = t.linalg.cross(e1, e2, dim=-1)     # (N, 3) — un-normalized
norms = n.norm(dim=-1, keepdim=True)   # (N, 1)
```

**Degeneracy.** A triangle is degenerate iff its three vertices are colinear ⇒ `cross == 0` ⇒ `||n|| == 0` ⇒ division by zero produces `nan` / `inf`. Real renderers test `norms > eps` (eps ~ 1e-8) and either skip the triangle or substitute a sentinel normal.

**Returning a mask is better than skipping.** A boolean `valid: (N,)` lets downstream code decide what to do (skip in lighting, but maybe keep for connectivity).

### Exercise 2 — batched unit normals + degenerate-triangle mask

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.linalg.cross(..., dim=-1)` over a `(N, 3, 3)` batch of triangle vertices and produce both the unit-normals tensor AND a boolean mask of degenerate (colinear) triangles.
> Keywords: cross-product, batched, degenerate, mask
> ```

**KCs targeted:** `batched-cross-along-dim-minus-1`, `degenerate-norm-mask`

Implement `ex2_batched_normals(tris, eps=1e-8)`. Compute unit surface normals for a batch of triangles AND flag the degenerate ones.

1. `tris` has shape `(N, 3, 3)` — N triangles, 3 vertices each, 3-D coords.
2. Edges from shared vertex `P1 = tris[:, 0]`:
   ```python
   e1 = tris[:, 1] - tris[:, 0]   # (N, 3)
   e2 = tris[:, 2] - tris[:, 0]   # (N, 3)
   ```
3. Cross product along the LAST axis: `n = t.linalg.cross(e1, e2, dim=-1)`. Shape `(N, 3)`.
4. Norms: `norms = n.norm(dim=-1, keepdim=True)` — shape `(N, 1)`.
5. Degenerate mask: `valid = (norms.squeeze(-1) > eps)` — shape `(N,)`, `True` iff the triangle has nonzero area.
6. Normalize SAFELY. For degenerate triangles, division would produce `nan` / `inf`; replace the denominator with `1.0` where the triangle is invalid (the resulting normal there is the zero vector — a sentinel that downstream code can detect).
   ```python
   safe_norms = norms.clamp(min=eps)
   unit = n / safe_norms
   # zero-out the degenerate ones so the value is a clean sentinel
   unit[~valid] = 0.0
   ```
7. Return `(unit, valid)`. `unit` has shape `(N, 3)`; `valid` has shape `(N,)` (bool).

**Do NOT** call `ex1_triangle_normal` in a Python loop. Use batched ops throughout.

Input: `tris` shape `(N, 3, 3)` float; `eps` float.
Output: tuple `(unit_normals (N,3), valid_mask (N,))`.

In [ ]:
def ex2_batched_normals(tris, eps=1e-8):
    e1 = tris[:, 1] - tris[:, 0]                   # (N, 3)
    e2 = tris[:, 2] - tris[:, 0]                   # (N, 3)
    n = t.linalg.cross(e1, e2, dim=-1)             # (N, 3)
    norms = n.norm(dim=-1, keepdim=True)           # (N, 1)
    valid = norms.squeeze(-1) > eps                # (N,) bool
    safe = norms.clamp(min=eps)
    unit = n / safe
    unit[~valid] = 0.0                             # sentinel
    return unit, valid


<details><summary>Solution</summary>

```python
def ex2_batched_normals(tris, eps=1e-8):
    e1 = tris[:, 1] - tris[:, 0]                   # (N, 3)
    e2 = tris[:, 2] - tris[:, 0]                   # (N, 3)
    n = t.linalg.cross(e1, e2, dim=-1)             # (N, 3)
    norms = n.norm(dim=-1, keepdim=True)           # (N, 1)
    valid = norms.squeeze(-1) > eps                # (N,) bool
    safe = norms.clamp(min=eps)
    unit = n / safe
    unit[~valid] = 0.0                             # sentinel
    return unit, valid
```

**Why `dim=-1` instead of `dim=1`.** Either works for a `(N, 3)` tensor. `dim=-1` is the convention that survives rank changes — the same code works if you later wrap everything in another batch dim (`(M, N, 3, 3)`).

**Why `clamp(min=eps)` then mask, not just `if-else`.** Branching per-element would require a Python loop or a `torch.where` chain. Clamping the denominator first ensures the division never produces `nan`/`inf`; then a single bool indexer (`unit[~valid] = 0.0`) replaces the bogus values with a clean sentinel. Vectorized + numerically safe.

**Why a zero vector as sentinel.** Downstream code can detect 'this normal is invalid' with `(unit == 0).all(-1)`, which is cheap to vectorize. A unit-length sentinel (e.g. +z) would confuse downstream lighting that takes `dot(L, n)` — a real, non-degenerate +z triangle would look identical to a degenerate one. The zero vector has no valid interpretation, so it's safely distinguishable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()